### Notebook
- **Name:** `cs_towers.ipynb`
- **Created/updated:** 2026-02-27
- **Python:** 3.x

### Purpose
Convert tower/support input data to GIS layers and standardized formats.

### Inputs
Excel file with supports `(x,y,z)` plus identifiers; config describing paths/CRS.

### Outputs
Shapefile (or equivalent GIS layer) for supports with attributes.

### Dependencies
- (see imports below)

### Usage
Executed by the project pipeline (e.g., via Papermill) or run interactively in Jupyter.

### Notes
- Keep paths and parameters centralized in `config.toml` / `CONFIG_PATH` where applicable.


In [ ]:
# Parameters (Papermill)
CONFIG_PATH = "" # r"E:\mario\trabajos2\viesgo_edp_portugal_cic\estudios_microclimaticos\Corredoria_Grado_1_y_2\config.toml" # "config.toml"
PROJECT_ROOT = r"E:\mario\python\criticalspam"

### Towers / supports — Excel to Shapefile

This notebook converts the Excel file containing tower/support coordinates `(x, y, z)` into a `.shp` file,
including the tower identifier (label/matriculation) as an attribute.


In [15]:
import sys
print(sys.version)
print(sys.executable)

3.12.9 (tags/v3.12.9:fdb8142, Feb  4 2025, 15:27:58) [MSC v.1942 64 bit (AMD64)]
c:\Python\Python312\python.exe


In [16]:
from __future__ import annotations

import re
import sys
from pathlib import Path
from typing import Optional

import re

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from pathlib import Path
from typing import Optional

from dataclasses import dataclass

import papermill as pm

import tomllib

### Configuration loading (example)

```python
with open(Path(CONFIG_PATH), "rb") as f:
    config_toml_path = tomllib.load(f)

print("towers_01.ipynb")
print("Config path:", config_toml_path)
```


In [17]:

# Ruta al directorio raíz del proyecto (sube 1 nivel desde notebooks/)
#ROOT = Path.cwd().resolve().parent

# Añade src/ al path de importación
SRC = Path(PROJECT_ROOT + r"\src") 
print(f"Adding to sys.path: {SRC}")

#SRC =r'E:\mario\python\criticalspam\src'

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# Ya puedes importar
from cs_utils import Config, join_base, load_config_toml

Adding to sys.path: E:\mario\python\criticalspam\src


In [18]:
cfg = load_config_toml(CONFIG_PATH)

print("cs_towers.ipynb")
print("Config path:", CONFIG_PATH)


cs_towers.ipynb
Config path: E:\mario\trabajos2\viesgo_edp_portugal_cic\estudios_microclimaticos\Corredoria_Grado_1_y_2\config.toml


In [19]:


def parse_number(value) -> float:
    """
    Convierte a float:
    - Si viene como número de Excel (int/float), lo convierte directamente.
    - Si viene como string, acepta:
        1.234.567,89  (EU)
        1234567,89    (EU sin miles)
        1234567.89    (US)
        1.234.567     (miles)
    Nota: NO intenta “inventar” decimales: eso se hace en la fase de escalado.
    """
    if value is None or (isinstance(value, float) and pd.isna(value)):
        raise ValueError("Valor vacío/NaN")

    # Si ya es numérico (Excel), no hay separadores que interpretar
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return float(value)

    s = str(value).strip()
    if s == "":
        raise ValueError("Cadena vacía")

    s = s.replace(" ", "").replace("'", "")

    # EU: punto miles + coma decimal
    if "," in s and "." in s:
        return float(s.replace(".", "").replace(",", "."))

    # EU: coma decimal
    if "," in s and "." not in s:
        return float(s.replace(",", "."))

    # Solo puntos: si hay >1, asume miles
    if "." in s and s.count(".") > 1:
        return float(s.replace(".", ""))

    # Resto: float estándar
    return float(s)


def parse_xyz_with_autoscale(value, kind: str) -> float:
    """
    Convierte a float y corrige el caso típico de Excel “sin decimales”
    donde el valor real está en milésimas (×1/1000), eligiendo la versión
    que cae en el rango plausible para cada coordenada.

    kind: 'x', 'y', 'z'
    """
    v = parse_number(value)

    # Rangos plausibles (ETRS89/UTM): ajusta si procede
    ranges = {
        "x": (100_000.0, 900_000.0),
        "y": (0.0, 10_000_000.0),
        "z": (-500.0, 9_000.0),
    }
    lo, hi = ranges[kind]

    # Si ya está en rango, OK
    if lo <= v <= hi:
        return v

    # Probar escalado milésimas (muy típico en estos ficheros)
    v1 = v / 1000.0
    if lo <= v1 <= hi:
        return v1

    # (Opcional) Si trabajases con centésimas, podrías habilitar /100, pero aquí no lo activo para no “inventar”.
    # v2 = v / 100.0

    return v  # si no encaja, devuelve el valor original (para detectar anomalías)


def excel_to_shp(
    xlsx_path: Path,
    shp_path: Path,
    sheet: Optional[str] = None,
    epsg: Optional[int] = None,
    col_x: str = "X",
    col_y: str = "Y",
    col_z: str = "Z",
    col_label: str = "Structure Comment",
) -> pd.DataFrame:
    xlsx_path = Path(xlsx_path)
    shp_path = Path(shp_path)

    wb = pd.read_excel(xlsx_path, sheet_name=sheet, engine="openpyxl")  # no forzamos str: Excel puede traer numérico
    if isinstance(wb, dict):
        if not wb:
            raise ValueError("El Excel no contiene hojas.")
        df = wb[next(iter(wb))]
    else:
        df = wb

    df.columns = [str(c).strip() for c in df.columns]

    for c in (col_x, col_y, col_z, col_label):
        if c not in df.columns:
            raise KeyError(f"Falta la columna '{c}'. Columnas disponibles: {list(df.columns)}")

    # Conversión + autoscale
    df["_X"] = df[col_x].apply(lambda v: parse_xyz_with_autoscale(v, "x"))
    df["_Y"] = df[col_y].apply(lambda v: parse_xyz_with_autoscale(v, "y"))
    df["_Z"] = df[col_z].apply(lambda v: parse_xyz_with_autoscale(v, "z"))

    # Campo para etiquetar
    df["MAT"] = df[col_label].astype(str).str.strip()

    # Diagnóstico rápido (puedes comentar luego)
    print("Rangos interpretados:")
    print("X:", df["_X"].min(), df["_X"].max())
    print("Y:", df["_Y"].min(), df["_Y"].max())
    print("Z:", df["_Z"].min(), df["_Z"].max())

    geometry = [Point(x, y, z) for x, y, z in zip(df["_X"], df["_Y"], df["_Z"])]
    gdf = gpd.GeoDataFrame(df.drop(columns=["_X", "_Y", "_Z"]), geometry=geometry)

    if epsg is not None:
        gdf = gdf.set_crs(epsg=epsg, allow_override=True)

    shp_path.parent.mkdir(parents=True, exist_ok=True)
    gdf.to_file(shp_path, driver="ESRI Shapefile", encoding="UTF-8")
    
    return df


In [20]:

def build_vanos_df(
    df: pd.DataFrame,
    mat_col: str = "MAT",
    x_col: str = "_X",
    y_col: str = "_Y",
    z_col: str = "_Z",
) -> pd.DataFrame:
    """
    Devuelve un DataFrame con:
    MAT, UTMx, UTMy, UTMz, distancia, direccion, vanoUTMx, vanoUTMy

    - distancia: sqrt((Δx)^2 + (Δy)^2) entre fila i y i-1
    - direccion: azimut (deg) desde Norte (+Y) horario, en [0, 360)
    - vanoUTMx/vanoUTMy: punto medio entre (x_i,y_i) y (x_{i-1},y_{i-1})
    """
    # Copias como float para asegurar operaciones numéricas
    x = df[x_col].astype(float)
    y = df[y_col].astype(float)
    z = df[z_col].astype(float)

    dx = x.diff()
    dy = y.diff()

    distancia = np.hypot(dx, dy)

    # Azimut desde Norte: atan2(Δx, Δy)  -> radianes, luego grados
    direccion = (np.degrees(np.arctan2(dx, dy)) + 360.0) % 360.0

    vanoUTMx = (x + x.shift(1)) / 2.0
    vanoUTMy = (y + y.shift(1)) / 2.0

    out = pd.DataFrame({
        "MAT": df[mat_col],
        "UTMx": x,
        "UTMy": y,
        "UTMz": z,
        "distancia": distancia,
        "direccion": direccion,
        "vanoUTMx": vanoUTMx,
        "vanoUTMy": vanoUTMy,
    })

    return out


In [21]:
in_xlsx = cfg.in_xlsx  #Path(r'E:\mario\trabajos2\viesgo_edp_portugal_cic\estudios_microclimaticos\Corredoria_Grado_1_y_2\Apoyos Corredoria-Grado.xlsx')
out_apoyos_shp = cfg.out_apoyos_shp # Path(r'E:\mario\trabajos2\viesgo_edp_portugal_cic\estudios_microclimaticos\Corredoria_Grado_1_y_2\Apoyos Corredoria-Grado.shp')
apoyos_epsg_arg = cfg.apoyos_epsg_arg  # 25830

apoyos_df = excel_to_shp(in_xlsx, out_apoyos_shp) #, epsg=apoyos_epsg_arg)
print(f"OK: generado {out_apoyos_shp}")

Rangos interpretados:
X: 257521.932 270789.686
Y: 4806681.225 4808504.318
Z: 86.685 380.887
OK: generado E:\mario\trabajos2\viesgo_edp_portugal_cic\estudios_microclimaticos\Corredoria_Grado_1_y_2\Apoyos\Apoyos Corredoria-Grado.shp


C:\Users\mananam\AppData\Local\Temp\ipykernel_84456\1556517123.py:122: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(shp_path, driver="ESRI Shapefile", encoding="UTF-8")


In [22]:
apoyos_df.head()

,Number,Station,X,Y,Z,Line angle,Structure Comment,_X,_Y,_Z,MAT
0,1,NaN,257521932,4808272963,107672,NaN,Pórtico Corredoria,257521.932,4808272.963,107.672,Pórtico Corredoria
1,2,186270.0,257702460,4808318851,115947,0.0197,AP111280,257702.460,4808318.851,115.947,AP111280
2,3,551326.0,258056294,4808408674,100068,0.0783,AP300628,258056.294,4808408.674,100.068,AP300628
3,4,941933.0,258435010,4808504318,105202,243858.0000,AP300627,258435.010,4808504.318,105.202,AP300627
4,5,1118486.0,258609941,4808480437,86685,0.2709,AP300626,258609.941,4808480.437,86.685,AP300626


In [23]:
vanos_df = build_vanos_df(apoyos_df)

vanos_df.head()

print(f"vanos_df tiene {len(vanos_df)} filas. Exportando puntos intermedios de vano a shapefile...  ")
print(f"cfg.vanos_midpoints_shp: {cfg.out_vanos_shp}")
print(f"epsg: {cfg.apoyos_epsg_arg}")

vanos_df tiene 57 filas. Exportando puntos intermedios de vano a shapefile...  
cfg.vanos_midpoints_shp: E:\mario\trabajos2\viesgo_edp_portugal_cic\estudios_microclimaticos\Corredoria_Grado_1_y_2\Calculos\Corredoria_Grado_1_y_2_vanos.shp
epsg: 25830


In [24]:
print(f"vanos_df tiene {len(vanos_df)} filas. Exportando puntos intermedios de vano a shapefile...  ")
print(f"cfg.vanos_midpoints_shp: {cfg.out_vanos_shp}")
print(f"epsg: {cfg.apoyos_epsg_arg}")

vanos_df tiene 57 filas. Exportando puntos intermedios de vano a shapefile...  
cfg.vanos_midpoints_shp: E:\mario\trabajos2\viesgo_edp_portugal_cic\estudios_microclimaticos\Corredoria_Grado_1_y_2\Calculos\Corredoria_Grado_1_y_2_vanos.shp
epsg: 25830


In [25]:


def export_vanos_midpoints_shp(
    vanos_df: pd.DataFrame,
    shp_path: str | Path,
    epsg: int = 25830,
    xmid_col: str = "vanoUTMx",
    ymid_col: str = "vanoUTMy",
    keep_cols: list[str] | None = None,
) -> Path:
    """
    Exporta a Shapefile (QGIS) los puntos intermedios de vano.

    Nota: evita duplicados si keep_cols ya incluye xmid_col/ymid_col.
    """
    import geopandas as gpd

    shp_path = Path(shp_path)
    shp_path.parent.mkdir(parents=True, exist_ok=True)

    # Filtra filas sin vano (primera fila) o con NaNs
    gdf_src = vanos_df.copy().dropna(subset=[xmid_col, ymid_col])

    # Selección de atributos sin duplicar nombres
    if keep_cols is not None:
        cols = list(keep_cols)
        if xmid_col not in cols:
            cols.append(xmid_col)
        if ymid_col not in cols:
            cols.append(ymid_col)

        missing = [c for c in cols if c not in gdf_src.columns]
        if missing:
            raise KeyError(f"Columnas no encontradas en vanos_df: {missing}")

        gdf_src = gdf_src.loc[:, cols]

    # Asegura Series 1D aunque existan columnas duplicadas por error upstream
    xmid = gdf_src.loc[:, xmid_col]
    ymid = gdf_src.loc[:, ymid_col]
    if isinstance(xmid, pd.DataFrame):
        xmid = xmid.iloc[:, 0]
    if isinstance(ymid, pd.DataFrame):
        ymid = ymid.iloc[:, 0]

    gdf = gpd.GeoDataFrame(
        gdf_src,
        geometry=gpd.points_from_xy(xmid.astype(float), ymid.astype(float)),
        crs=f"EPSG:{int(epsg)}",
    )

    # Shapefile: nombres de campo limitados (~10 chars)
    rename_map = {}
    if xmid_col in gdf.columns and len(xmid_col) > 10:
        rename_map[xmid_col] = "VANO_X"
    if ymid_col in gdf.columns and len(ymid_col) > 10:
        rename_map[ymid_col] = "VANO_Y"
    if rename_map:
        gdf = gdf.rename(columns=rename_map)

    gdf.to_file(shp_path, driver="ESRI Shapefile", encoding="UTF-8")
    return shp_path


In [26]:
out_shp = export_vanos_midpoints_shp(
    vanos_df,
    cfg.out_vanos_shp,
    epsg=int(cfg.apoyos_epsg_arg),
    keep_cols=["MAT", "distancia", "direccion", "vanoUTMx", "vanoUTMy"]
)
print(out_shp)

E:\mario\trabajos2\viesgo_edp_portugal_cic\estudios_microclimaticos\Corredoria_Grado_1_y_2\Calculos\Corredoria_Grado_1_y_2_vanos.shp
